# Local RAG on Google Colab
Free stack: Ollama (Llama 3.2) + sentence-transformers + ChromaDB.

**Tip:** Runtime > Change runtime type > T4 GPU makes answers much faster.

## 1. Install Ollama and Python packages

In [ ]:
!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q chromadb sentence-transformers ollama pypdf

## 2. Start the Ollama server and download the model

In [ ]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull llama3.2

## 3. Upload your documents (PDF, TXT, MD)

In [ ]:
import os
from google.colab import files

os.makedirs("docs", exist_ok=True)
for name, data in files.upload().items():
    with open(f"docs/{name}", "wb") as f:
        f.write(data)
print(os.listdir("docs"))

## 4. RAG code

In [ ]:
from pathlib import Path
import chromadb, ollama
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "llama3.2"
CHUNK_SIZE, OVERLAP, TOP_K = 800, 150, 4
SUPPORTED = {".pdf", ".txt", ".md"}

embedder = SentenceTransformer(EMBED_MODEL)
db = chromadb.PersistentClient(path="chroma_db")
col = db.get_or_create_collection("docs", metadata={"hnsw:space": "cosine"})

def read_file(path):
    if path.suffix.lower() == ".pdf":
        return "\n".join(p.extract_text() or "" for p in PdfReader(str(path)).pages)
    return path.read_text(encoding="utf-8", errors="ignore")

def chunk(text):
    step = CHUNK_SIZE - OVERLAP
    chunks = [text[i:i + CHUNK_SIZE].strip() for i in range(0, len(text), step)]
    return [c for c in chunks if c]

def ingest(folder):
    for path in Path(folder).rglob("*"):
        if path.suffix.lower() not in SUPPORTED:
            continue
        chunks = chunk(read_file(path))
        if not chunks:
            print(f"Skipped {path} (no extractable text)")
            continue
        col.upsert(
            ids=[f"{path}:{i}" for i in range(len(chunks))],
            documents=chunks,
            embeddings=embedder.encode(chunks).tolist(),
            metadatas=[{"source": str(path)}] * len(chunks),
        )
        print(f"Indexed {path} ({len(chunks)} chunks)")

def ask(question):
    q_emb = embedder.encode([question]).tolist()
    hits = col.query(query_embeddings=q_emb, n_results=TOP_K)
    context = "\n\n---\n\n".join(hits["documents"][0])
    prompt = (
        "Answer using ONLY the context below. "
        "If the answer isn't there, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    reply = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}])
    print(reply["message"]["content"])
    print("\nSources:", {m["source"] for m in hits["metadatas"][0]})

## 5. Index and ask

In [ ]:
ingest("docs")

In [ ]:
ask("Who did Emma Watson co-found Renais with?")